In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iambahar/mimimiinino/MI_NonMI_Patient_Dataset.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/train_80_mode_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/train_80_mode_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/train_80_missing_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/val_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/test_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/tra

In [2]:
# ============================================================
# CARDIAC MI STAGING — BioMedBERT (Text-ified Tabular)
# TARGET: Phase (Non-MI, Chronic, Sub-acute, Acute)
# ============================================================
import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

def pip_install(pkg):
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"[OK] {pkg}")
    else:
        print(f"[WARN] {pkg}: {result.stderr[-200:]}")

pip_install("transformers")
pip_install("datasets")
pip_install("accelerate")
pip_install("evaluate")

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)
from datasets import Dataset

# ====================== CONFIG ======================
TRAIN_PATH = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx"
TEST_PATH  = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/test_10.xlsx"
TARGET_COL = "Phase"
RANDOM_SEED = 42

# ── Top-10 features from feature importance chart ──────────────────────────
SELECTED_FEATURES = [
    "Blood_Glucose_mmolL",
    "Diastolic_BP",
    "Chest_Pain",
    "Weighted_Risk_Score",
    "Systolic_BP",
    "Heart_Rate_bpm",
    "Weighted_Symptom_Score",
    "Serum_Creatinine_mgdL",
    "hs_Troponin_I_ngL",
    "Age_Adjusted_Troponin",
]
# ───────────────────────────────────────────────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Device: {DEVICE}")
print(f"Selected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}")

# ============================================================
# DATA LOADING & TEXT SERIALIZATION
# ============================================================
def load_and_serialize(train_path, test_path):
    train_df = pd.read_excel(train_path)
    test_df  = pd.read_excel(test_path)

    # ── Auto-rename target if missing in test set ──────────────────────────
    if TARGET_COL not in test_df.columns:
        for col in test_df.columns:
            if test_df[col].astype(str).isin(
                train_df[TARGET_COL].astype(str).unique()
            ).sum() > 5:
                test_df = test_df.rename(columns={col: TARGET_COL})
                print(f"[INFO] Auto-renamed target to '{TARGET_COL}'")
                break

    print(f"Train shape (raw): {train_df.shape} | Test shape (raw): {test_df.shape}")

    # ── Validate that all selected features exist ──────────────────────────
    missing_train = [f for f in SELECTED_FEATURES if f not in train_df.columns]
    missing_test  = [f for f in SELECTED_FEATURES if f not in test_df.columns]
    if missing_train:
        raise ValueError(f"Missing in train: {missing_train}")
    if missing_test:
        raise ValueError(f"Missing in test:  {missing_test}")

    # ── Keep only selected features + target ──────────────────────────────
    train_df = train_df[SELECTED_FEATURES + [TARGET_COL]]
    test_df  = test_df[SELECTED_FEATURES  + [TARGET_COL]]

    print(f"Train shape (filtered): {train_df.shape} | Test shape (filtered): {test_df.shape}")
    print(f"Train Phase distribution:\n{train_df[TARGET_COL].value_counts()}\n")

    # ── Encode labels ──────────────────────────────────────────────────────
    le = LabelEncoder()
    le.fit(train_df[TARGET_COL])
    y_train = le.transform(train_df[TARGET_COL])
    y_test  = le.transform(test_df[TARGET_COL])

    global CLASS_NAMES
    CLASS_NAMES = le.classes_.tolist()
    print(f"Classes: {CLASS_NAMES}")

    # ── Features only (target already encoded) ────────────────────────────
    X_train = train_df[SELECTED_FEATURES]
    X_test  = test_df[SELECTED_FEATURES]

    # ── Convert tabular row → natural text ────────────────────────────────
    def row_to_text(row):
        return " | ".join(
            [f"{col}:{val}" for col, val in row.items() if pd.notna(val)]
        )

    train_texts = X_train.apply(row_to_text, axis=1).tolist()
    test_texts  = X_test.apply(row_to_text, axis=1).tolist()

    # Preview
    print(f"\nSample serialized text:\n  {train_texts[0]}\n")

    return train_texts, test_texts, y_train, y_test, CLASS_NAMES


train_texts, test_texts, y_train, y_test, CLASS_NAMES = load_and_serialize(
    TRAIN_PATH, TEST_PATH
)

# ============================================================
# BioMedBERT SETUP
# ============================================================
print("\n" + "█"*70)
print(" MODEL: BioMedBERT")
print("█"*70)

MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Hugging Face Datasets ─────────────────────────────────────────────────
train_data = Dataset.from_dict({"text": train_texts, "label": y_train.tolist()})
test_data  = Dataset.from_dict({"text": test_texts,  "label": y_test.tolist()})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,   # 10 features → much shorter texts; 256 is sufficient
    )

train_data = train_data.map(tokenize_function, batched=True)
test_data  = test_data.map(tokenize_function, batched=True)

train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format("torch",  columns=["input_ids", "attention_mask", "label"])

# ── Model ─────────────────────────────────────────────────────────────────
num_labels = len(CLASS_NAMES)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# ====================== TRAINING ARGS ======================
training_args = TrainingArguments(
    output_dir="./biomedbert_results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=RANDOM_SEED,
    remove_unused_columns=False,
)

# ====================== METRICS ======================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
    }

# ====================== TRAINER ======================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# ====================== TRAINING ======================
print("Starting BioMedBERT fine-tuning on selected features...")
trainer.train()

# ====================== EVALUATION ======================
print("\nEvaluating on test set...")
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

# ── Final Predictions ─────────────────────────────────────────────────────
predictions = trainer.predict(test_data)
preds = np.argmax(predictions.predictions, axis=1)

print("\n" + "="*70)
print("BioMedBERT FINAL RESULTS  (top-10 features only)")
print("="*70)
print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_test, preds)
print("\nConfusion Matrix:\n", cm)

print("\n✅ BioMedBERT training & evaluation completed!")

[OK] transformers
[OK] datasets
[OK] accelerate
[OK] evaluate
Device: cuda
Selected features (10): ['Blood_Glucose_mmolL', 'Diastolic_BP', 'Chest_Pain', 'Weighted_Risk_Score', 'Systolic_BP', 'Heart_Rate_bpm', 'Weighted_Symptom_Score', 'Serum_Creatinine_mgdL', 'hs_Troponin_I_ngL', 'Age_Adjusted_Troponin']
Train shape (raw): (1776, 23) | Test shape (raw): (190, 23)
Train shape (filtered): (1776, 11) | Test shape (filtered): (190, 11)
Train Phase distribution:
Phase
Acute        520
Non-MI       434
Chronic      422
Sub-acute    400
Name: count, dtype: int64

Classes: ['Acute', 'Chronic', 'Non-MI', 'Sub-acute']

Sample serialized text:
  Blood_Glucose_mmolL:12.7 | Diastolic_BP:-0.2367988898819029 | Chest_Pain:1.0 | Weighted_Risk_Score:7.0 | Systolic_BP:-0.3695581856124019 | Heart_Rate_bpm:1.223287249716773 | Weighted_Symptom_Score:7.0 | Serum_Creatinine_mgdL:1.15 | hs_Troponin_I_ngL:1.774542798744036 | Age_Adjusted_Troponin:2.113646961457278


█████████████████████████████████████████████

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1776 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on test set...


Evaluation Results: {'eval_loss': 0.22118093073368073, 'eval_accuracy': 0.9578947368421052, 'eval_f1': 0.9586738227146815, 'eval_precision': 0.9600250626566416, 'eval_recall': 0.9578947368421052, 'eval_runtime': 1.8023, 'eval_samples_per_second': 105.422, 'eval_steps_per_second': 3.329, 'epoch': 5.0}

BioMedBERT FINAL RESULTS  (top-10 features only)
              precision    recall  f1-score   support

       Acute       0.95      0.92      0.94        65
     Chronic       1.00      1.00      1.00        52
      Non-MI       1.00      1.00      1.00        55
   Sub-acute       0.75      0.83      0.79        18

    accuracy                           0.96       190
   macro avg       0.93      0.94      0.93       190
weighted avg       0.96      0.96      0.96       190


Confusion Matrix:
 [[60  0  0  5]
 [ 0 52  0  0]
 [ 0  0 55  0]
 [ 3  0  0 15]]

✅ BioMedBERT training & evaluation completed!


In [3]:
# ==============================================================================
# PHASE 1: DATA CLEANING & REFERENCE ARTIFACT LOCKDOWN
# ==============================================================================
import numpy as np 
import pandas as pd 
import os
import joblib

# Load raw clinical dataset
file_path = '/kaggle/input/datasets/iambahar/mimimiinino/MI_NonMI_Patient_Dataset.xlsx'
df = pd.read_excel(file_path)

# 1. Parse Blood Pressure string coordinates
bp = df['Blood_Pressure_mmHg'].astype(str).str.extract(r'(\d+)/(\d+)')
df['Systolic_BP']  = pd.to_numeric(bp[0], errors='coerce')
df['Diastolic_BP'] = pd.to_numeric(bp[1], errors='coerce')
df.drop(columns=['Blood_Pressure_mmHg'], inplace=True)
print("[✓] Blood Pressure parsed into Systolic and Diastolic coordinates.")

# 2. Handle missingness structural deletions
drop_cols = [c for c in ['CK_MB_UL', 'BNP_pgmL'] if c in df.columns]
if drop_cols:
    df.drop(columns=drop_cols, inplace=True)

if 'hs_Troponin_I_ngL' in df.columns:
    df = df.dropna(subset=['hs_Troponin_I_ngL']).reset_index(drop=True)

# Define column layout schemas
NUM_COLS_RAW = ['Age', 'hs_Troponin_I_ngL', 'Blood_Glucose_mmolL', 'Serum_Creatinine_mgdL', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP']
NUM_COLS_RAW = [c for c in NUM_COLS_RAW if c in df.columns]

CAT_BIN_COLS = ['Sex', 'Chest_Pain', 'Shortness_of_Breath', 'Nausea', 'Sweating', 'Hypertension', 'Diabetes_Mellitus', 'Smoking', 'Hyperlipidaemia', 'Prior_MI', 'Family_History_CAD']

# 3. CRITICAL DEPLOYMENT ARTIFACT: Compute and Save Global Fallback References
# New incoming patient points won't have a known 'Phase', so we save global metrics for live fallback
imputation_references = {"numerical_medians": {}, "categorical_modes": {}}

for col in NUM_COLS_RAW:
    imputation_references["numerical_medians"][col] = float(df[col].median())
for col in CAT_BIN_COLS:
    mode_val = df[col].mode()
    imputation_references["categorical_modes"][col] = str(mode_val.iloc[0]) if not mode_val.empty else "Missing"

joblib.dump(imputation_references, 'training_imputation_references.joblib')
print("[SAVED] Global imputation fallbacks locked to disk.")

# Apply cross-sectional training imputation loops
df_mode = df.copy()
df_missing = df.copy()

for dataset in [df_mode, df_missing]:
    for col in NUM_COLS_RAW:
        if dataset[col].isna().sum() > 0:
            dataset[col] = dataset.groupby(['Phase', 'Sex'])[col].transform(lambda x: x.fillna(x.median()))
            dataset[col] = dataset.groupby('Phase')[col].transform(lambda x: x.fillna(x.median()))
            dataset[col] = dataset[col].fillna(imputation_references["numerical_medians"][col])

for col in CAT_BIN_COLS:
    df_mode[col] = df_mode.groupby(['Phase', 'Sex'])[col].transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x)
    df_mode[col] = df_mode.groupby('Phase')[col].transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x)
    df_mode[col] = df_mode[col].fillna(df_mode[col].mode().iloc[0])

for col in CAT_BIN_COLS:
    df_missing[col] = df_missing[col].fillna("Missing")

# ==============================================================================
# PHASE 2: PERSISTENT CATEGORICAL ENCODING & FEATURE ENGINEERING
# ==============================================================================
from sklearn.preprocessing import LabelEncoder

fitted_label_encoders = {}

def encode_and_persist(df_target, num_cols, cat_cols):
    df_res = df_target.copy()
    for col in num_cols:
        df_res[col] = pd.to_numeric(df_res[col], errors="coerce").astype(float)
    for col in cat_cols:
        if df_res[col].dtype == object or str(df_res[col].dtype).startswith('str'):
            le = LabelEncoder()
            nan_mask = df_res[col].isna()
            df_res[col] = le.fit_transform(df_res[col].astype(str))
            df_res.loc[nan_mask, col] = np.nan
            fitted_label_encoders[col] = le
        df_res[col] = pd.to_numeric(df_res[col], errors="coerce")
    return df_res

df_mode_encoded = encode_and_persist(df_mode, NUM_COLS_RAW, CAT_BIN_COLS)
joblib.dump(fitted_label_encoders, 'fitted_categorical_encoders.joblib')
print("[SAVED] Categorical label encoder mappings successfully saved.")

# Complete Feature Engineering pipeline steps
SYMPTOM_WEIGHTS = {'Chest_Pain': 4, 'Shortness_of_Breath': 1, 'Nausea': 1, 'Sweating': 1}
RISK_WEIGHTS = {'Prior_MI': 1, 'Hypertension': 1, 'Diabetes_Mellitus': 1, 'Smoking': 1, 'Hyperlipidaemia': 1, 'Family_History_CAD': 1}

df_fe = df_mode_encoded.copy()
df_fe['Pulse_Pressure'] = df_fe['Systolic_BP'] - df_fe['Diastolic_BP']
df_fe['Weighted_Symptom_Score'] = sum(df_fe[col] * w for col, w in SYMPTOM_WEIGHTS.items())
df_fe['Weighted_Risk_Score'] = sum(df_fe[col] * w for col, w in RISK_WEIGHTS.items())
df_fe['Age_Adjusted_Troponin'] = df_fe['Age'] * np.log1p(df_fe['hs_Troponin_I_ngL'])

# ==============================================================================
# PHASE 3: QUANTILE NORMALIZATION CHECKPOINT & SAMPLING SPLITS
# ==============================================================================
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

continuous_cols = ['Age', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP', 'hs_Troponin_I_ngL', 'Pulse_Pressure', 'Age_Adjusted_Troponin']

qt = QuantileTransformer(output_distribution='normal', random_state=42)
df_fe[continuous_cols] = qt.fit_transform(df_fe[continuous_cols])
joblib.dump(qt, 'fitted_quantile_transformer.joblib')
print("[SAVED] Continuous feature Quantile Normalization transformer locked.")

# Partition validation matrices
train_df, temp_df = train_test_split(df_fe, test_size=0.20, stratify=df_fe["Phase"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["Phase"], random_state=42)

# ==============================================================================
# PHASE 4: SMOTE OVERSAMPLING & ENSEMBLE FEATURE SELECTION
# ==============================================================================
from imblearn.over_sampling import SMOTE
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, chi2, f_classif

feature_cols = [c for c in train_df.columns if c != "Phase"]
X_train = train_df[feature_cols].astype(float)
y_train = train_df["Phase"]

# Run class-imbalance oversampling adjustments
class_counts = y_train.value_counts().to_dict()
sampling_strategy = {"Non-MI": class_counts.get("Non-MI", 0), "Chronic": class_counts.get("Chronic", 0), "Sub-acute": 400, "Acute": class_counts.get("Acute", 0)}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=min(5, min(class_counts.values()) - 1))
X_res, y_res = smote.fit_resample(X_train, y_train)

# Persist Target Class Mappings and Input Schema Configurations
le_target = LabelEncoder()
y_res_encoded = le_target.fit_transform(y_res)
joblib.dump(le_target, 'fitted_target_phase_encoder.joblib')
joblib.dump(feature_cols, 'training_features_schema_list.joblib')
print("[SAVED] Target encoder matrix and structural input feature layouts locked.")

# Feature Ranking Evaluation (Chi2, ANOVA, RFE, SHAP)
X_pos = X_res - X_res.min()
chi2_scores, _ = chi2(X_pos, y_res_encoded)
anova_scores, _ = f_classif(X_res, y_res_encoded)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=1, step=1).fit(X_res, y_res_encoded)
rf_full = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_res, y_res_encoded)

explainer = shap.TreeExplainer(rf_full)
shap_values = explainer.shap_values(X_res)
shap_mean = np.mean([np.abs(v).mean(axis=0) for v in shap_values], axis=0) if isinstance(shap_values, list) else np.abs(shap_values).mean(axis=(0, 2))

print("\n" + "="*60)
print(" Pipeline Processing Complete. All Inference Checkpoints Secured.")
print("="*60)


# ==============================================================================
# PHASE 5: LIVE REAL-TIME INFERENCE SINGLE-PATIENT WRAPPER
# ==============================================================================
def preprocess_single_point(raw_data_dict, artifacts_dir="."):
    """
    Transforms a single raw incoming clinical patient dictionary observation point 
    using the exact locked pipeline parameters extracted during original training.
    """
    refs = joblib.load(f"{artifacts_dir}/training_imputation_references.joblib")
    encoders = joblib.load(f"{artifacts_dir}/fitted_categorical_encoders.joblib")
    quantile_transformer = joblib.load(f"{artifacts_dir}/fitted_quantile_transformer.joblib")
    expected_features = joblib.load(f"{artifacts_dir}/training_features_schema_list.joblib")
    
    symptom_weights = {'Chest_Pain': 4, 'Shortness_of_Breath': 1, 'Nausea': 1, 'Sweating': 1}
    risk_weights = {'Prior_MI': 1, 'Hypertension': 1, 'Diabetes_Mellitus': 1, 'Smoking': 1, 'Hyperlipidaemia': 1, 'Family_History_CAD': 1}
    
    proc = raw_data_dict.copy()
    if "Blood_Pressure_mmHg" in proc:
        try:
            sys_val, dia_val = str(proc["Blood_Pressure_mmHg"]).split('/')
            proc["Systolic_BP"] = float(sys_val)
            proc["Diastolic_BP"] = float(dia_val)
        except Exception:
            proc["Systolic_BP"] = np.nan
            proc["Diastolic_BP"] = np.nan
        del proc["Blood_Pressure_mmHg"]

    for col in ["CK_MB_UL", "BNP_pgmL"]:
        if col in proc: del proc[col]
            
    # Apply continuous/categorical default fallbacks safely
    for col, fallback_median in refs["numerical_medians"].items():
        if col not in proc or pd.isna(proc[col]): proc[col] = fallback_median
    for col, fallback_mode in refs["categorical_modes"].items():
        if col not in proc or pd.isna(proc[col]) or proc[col] == "": proc[col] = fallback_mode

    # Run persistent encoders
    for col, encoder_obj in encoders.items():
        if col in proc:
            val_str = str(proc[col])
            proc[col] = int(encoder_obj.transform([val_str])[0]) if val_str in encoder_obj.classes_ else int(encoder_obj.transform([encoder_obj.classes_[0]])[0])

    single_row_df = pd.DataFrame([proc])

    # Dynamic feature engineering
    single_row_df['Pulse_Pressure'] = single_row_df['Systolic_BP'] - single_row_df['Diastolic_BP']
    single_row_df['Weighted_Symptom_Score'] = sum(single_row_df[col] * w for col, w in symptom_weights.items())
    single_row_df['Weighted_Risk_Score'] = sum(single_row_df[col] * w for col, w in risk_weights.items())
    single_row_df['Age_Adjusted_Troponin'] = single_row_df['Age'] * np.log1p(single_row_df['hs_Troponin_I_ngL'])

    # Map onto exact continuous normal distributions from training session
    continuous_layout = ['Age', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP', 'hs_Troponin_I_ngL', 'Pulse_Pressure', 'Age_Adjusted_Troponin']
    single_row_df[continuous_layout] = quantile_transformer.transform(single_row_df[continuous_layout])

    return single_row_df[expected_features].astype(float)




[✓] Blood Pressure parsed into Systolic and Diastolic coordinates.
[SAVED] Global imputation fallbacks locked to disk.
[SAVED] Categorical label encoder mappings successfully saved.
[SAVED] Continuous feature Quantile Normalization transformer locked.
[SAVED] Target encoder matrix and structural input feature layouts locked.

 Pipeline Processing Complete. All Inference Checkpoints Secured.


In [4]:
# Simulating a new, un-preprocessed Non-MI patient hitting the server
def predict_phase_with_biomedbert(processed_matrix_df, model, tokenizer, class_names, device="cuda"):
    """
    Takes the processed dataframe matrix output from preprocess_single_point,
    filters the top 10 features, serializes them to a text string,
    and runs inference through BioMedBERT.
    """
    model.eval()
    
    # 1. Define the top 10 features exactly as your ensemble ranked them
    selected_features = [
        "Blood_Glucose_mmolL",
        "Diastolic_BP",
        "Chest_Pain",
        "Weighted_Risk_Score",
        "Systolic_BP",
        "Heart_Rate_bpm",
        "Weighted_Symptom_Score",
        "Serum_Creatinine_mgdL",
        "hs_Troponin_I_ngL",
        "Age_Adjusted_Troponin"
    ]
    
    # 2. Extract the single row as a dictionary series
    patient_row = processed_matrix_df.iloc[0]
    
    # 3. Filter down and keep only the top 10 columns in order
    filtered_features = {f: patient_row[f] for f in selected_features if f in patient_row}
    
    # 4. Serialize the normalized coordinates into text format
    # Using float formatting to keep the string tokens clean for BioMedBERT
    serialized_text = " | ".join([f"{col}:{val:.4f}" for col, val in filtered_features.items()])
    print("\n" + "="*70)
    print(" GENERATED INFERENCE TEXT FOR BIOMEDBERT")
    print("="*70)
    print(f"{serialized_text}\n")
    
    # 5. Tokenize text input sequence
    inputs = tokenizer(
        serialized_text, 
        padding="max_length", 
        truncation=True, 
        max_length=256, 
        return_tensors="pt"
    ).to(device)
    
    # 6. Run model forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        
    # 7. Match max probability to string label
    predicted_class_idx = np.argmax(probabilities)
    predicted_label = class_names[predicted_class_idx]
    
    # 8. Compile confidence breakdown
    confidence_breakdown = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}
    
    print(f"Diagnostic Decision -> [ {predicted_label} ]")
    print("\nConfidence Matrix:")
    for phase, prob in confidence_breakdown.items():
        print(f"  {phase:<12}: {prob * 100:.2f}%")
        
    return predicted_label, confidence_breakdown

# ==============================================================================
# EXECUTING THE COMPLETE INFRASTRUCTURE PIPELINE LOOP
# ==============================================================================

In [5]:


sample_raw_patient = {
    "Age": 44,
    "Sex": "Female",
    "Chest_Pain": "Yes",
    "Shortness_of_Breath": "No",
    "Nausea": "No",
    "Sweating": "Yes",
    "hs_Troponin_I_ngL": 0.0063,        # Completely normal baseline troponin
    "Blood_Glucose_mmolL": 8.6,
    "Serum_Creatinine_mgdL": 1.03,
    "Hypertension": "Yes",
    "Diabetes_Mellitus": "Yes",
    "Smoking": "No",
    "Hyperlipidaemia": "No",
    "Prior_MI": "No",
    "Family_History_CAD": "Yes",
    "Blood_Pressure_mmHg": "124/85",    # Near-normal blood pressure
    "Heart_Rate_bpm": 81                # Normal resting heart rate
}
# 1. Process the raw patient entry into the standardized matrix format
processed_point = preprocess_single_point(sample_raw_patient, artifacts_dir=".")

# 2. Pass the processed matrix straight to your Transformer for diagnostic categorization
predicted_phase, scores = predict_phase_with_biomedbert(
    processed_matrix_df=processed_point,
    model=model,
    tokenizer=tokenizer,
    class_names=CLASS_NAMES,
    device=DEVICE
)


 GENERATED INFERENCE TEXT FOR BIOMEDBERT
Blood_Glucose_mmolL:8.6000 | Diastolic_BP:-0.8625 | Chest_Pain:1.0000 | Weighted_Risk_Score:3.0000 | Systolic_BP:-1.1998 | Heart_Rate_bpm:-1.6347 | Weighted_Symptom_Score:5.0000 | Serum_Creatinine_mgdL:1.0300 | hs_Troponin_I_ngL:-1.6300 | Age_Adjusted_Troponin:-1.6138

Diagnostic Decision -> [ Non-MI ]

Confidence Matrix:
  Acute       : 0.04%
  Chronic     : 0.07%
  Non-MI      : 99.87%
  Sub-acute   : 0.03%


In [6]:
import sys
!{sys.executable} -m pip install -q dice-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.3 MB/s eta 0:00:00


# Non-MI to Acute

In [7]:
import dice_ml
import pandas as pd
import numpy as np
import torch
import joblib

# ==============================================================================
# 1. PREDICTION FUNCTION WITH ROBUST TYPE CASTING
# ==============================================================================
def dice_transformer_predict_fn(input_df):
    """
    Acts as a black-box bridge. DiCE inputs a tabular DataFrame,
    isolates the 10 features, serializes them to text, and runs BioMedBERT.
    """
    model.eval()
    
    selected_features = [
        "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
        "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
        "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
    ]
    
    probabilities_list = []
    
    for _, row in input_df.iterrows():
        serialized_text = " | ".join([f"{col}:{float(row[col]):.4f}" for col in selected_features])
        
        inputs = tokenizer(
            serialized_text, 
            padding="max_length", 
            truncation=True, 
            max_length=256, 
            return_tensors="pt"
        ).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(**inputs)
            prob = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
            probabilities_list.append(prob)
            
    return np.array(probabilities_list)


# ==============================================================================
# 2. SKLEARN-COMPATIBLE WRAPPER CLASS
# ==============================================================================
class BioMedBERTWrapper:
    """
    Wraps the transformer predict function to mimic sklearn's classifier interface.
    DiCE's sklearn backend requires predict_proba() and predict() methods.
    """
    def predict_proba(self, input_df):
        if not isinstance(input_df, pd.DataFrame):
            input_df = pd.DataFrame(input_df, columns=selected_features_list)
        input_df = input_df.astype(float)
        return dice_transformer_predict_fn(input_df)
    
    def predict(self, input_df):
        proba = self.predict_proba(input_df)
        return np.argmax(proba, axis=1)


# ==============================================================================
# 3. INITIALIZE DATA AND CONFIGURATION ALIGNMENTS
# ==============================================================================
selected_features_list = [
    "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
    "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
    "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
]

# A. Background Pool DataFrame MUST contain the outcome column
test_pool_df = test_df[selected_features_list].copy().astype(float)
test_pool_df['Phase'] = test_df['Phase'].apply(lambda x: CLASS_NAMES.index(x)).astype(int)

# B. Query Instance DataFrame must NOT contain the outcome column
query_instance_df = processed_point[selected_features_list].copy().astype(float)

# Map the background dataset properties into DiCE
d = dice_ml.Data(
    dataframe=test_pool_df, 
    continuous_features=[col for col in selected_features_list if col != "Chest_Pain"],
    outcome_name='Phase'
)

m = dice_ml.Model(
    model=BioMedBERTWrapper(),
    backend="sklearn",
    model_type="classifier"
)

exp = dice_ml.Dice(d, m, method="genetic")


# ==============================================================================
# 4. GENERATE COUNTERFACTUAL RECIPES
# ==============================================================================
print("\n" + "="*80)
print(" SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS...")
print("="*80)

target_class_index = CLASS_NAMES.index("Acute")

cf = exp.generate_counterfactuals(
    query_instances=query_instance_df,
    total_CFs=15,
    desired_class=target_class_index,
    proximity_weight=0.5,                 
    diversity_weight=1.5
)

cf.visualize_as_dataframe(show_only_changes=True)


# ==============================================================================
# 5. COUNTERFACTUAL DECODER — Reverses preprocessing back to clinical units
# ==============================================================================
def decode_counterfactuals(cf_object, raw_patient_dict, artifacts_dir="."):
    """
    Decodes DiCE counterfactuals from scaled/engineered space back to
    clinically interpretable raw values.

    Args:
        cf_object        : The DiCE counterfactual result object from generate_counterfactuals()
        raw_patient_dict : The original raw patient dictionary (sample_raw_patient)
        artifacts_dir    : Directory where joblib artifacts are saved

    Returns:
        A human-readable summary DataFrame with original vs counterfactual values + delta
    """

    # ------------------------------------------------------------------
    # A. Load all locked training artifacts
    # ------------------------------------------------------------------
    qt        = joblib.load(f"{artifacts_dir}/fitted_quantile_transformer.joblib")
    encoders  = joblib.load(f"{artifacts_dir}/fitted_categorical_encoders.joblib")

    # Columns the QuantileTransformer was fitted on — ORDER MUST MATCH TRAINING
    continuous_layout = [
        'Age', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP',
        'hs_Troponin_I_ngL', 'Pulse_Pressure', 'Age_Adjusted_Troponin'
    ]

    SYMPTOM_WEIGHTS = {'Chest_Pain': 4, 'Shortness_of_Breath': 1, 'Nausea': 1, 'Sweating': 1}
    RISK_WEIGHTS    = {'Prior_MI': 1, 'Hypertension': 1, 'Diabetes_Mellitus': 1,
                       'Smoking': 1, 'Hyperlipidaemia': 1, 'Family_History_CAD': 1}

    # ------------------------------------------------------------------
    # B. Extract processed query + counterfactual rows from DiCE object
    # ------------------------------------------------------------------
    cf_df   = cf_object.cf_examples_list[0].final_cfs_df.copy()
    orig_df = cf_object.cf_examples_list[0].test_instance_df.copy()
    orig_row = orig_df[selected_features_list].iloc[0]

    # ------------------------------------------------------------------
    # C. Pre-compute the scaled Age value from raw age using the qt
    #    (Age is not in selected_features_list but needed for qt inverse)
    # ------------------------------------------------------------------
    age_raw = float(raw_patient_dict["Age"])
    # Build a dummy row with only Age varying, inverse rest doesn't matter
    dummy_for_age = np.zeros((1, len(continuous_layout)))
    dummy_for_age[0][continuous_layout.index('Age')] = qt.transform(
        pd.DataFrame([[age_raw, 0, 0, 0, 0, 0, 0]], columns=continuous_layout)
    )[0][continuous_layout.index('Age')]
    scaled_age = qt.transform(
        pd.DataFrame([[age_raw, 0, 0, 0, 0, 0, 0]], columns=continuous_layout)
    )[0][0]

    # ------------------------------------------------------------------
    # D. Helper: safely get CF value or fall back to original scaled value
    # ------------------------------------------------------------------
    def get_val(row_dict, col, fallback):
        v = row_dict.get(col, None)
        if v is None:
            return float(fallback)
        if isinstance(v, float) and np.isnan(v):
            return float(fallback)
        try:
            f = float(v)
            return float(fallback) if np.isnan(f) else f
        except (ValueError, TypeError):
            return float(fallback)

    # ------------------------------------------------------------------
    # E. Helper: inverse-transform one CF row back to raw clinical values
    # ------------------------------------------------------------------
    def inverse_transform_row(row_dict):
        scaled_sys  = get_val(row_dict, "Systolic_BP",      orig_row["Systolic_BP"])
        scaled_dia  = get_val(row_dict, "Diastolic_BP",     orig_row["Diastolic_BP"])
        scaled_hr   = get_val(row_dict, "Heart_Rate_bpm",   orig_row["Heart_Rate_bpm"])
        scaled_trop = get_val(row_dict, "hs_Troponin_I_ngL",orig_row["hs_Troponin_I_ngL"])
        scaled_aat  = get_val(row_dict, "Age_Adjusted_Troponin", orig_row["Age_Adjusted_Troponin"])

        # Pulse_Pressure not in selected_features — approximate from scaled sys/dia
        # Best available proxy since we don't store it separately post-DiCE
        scaled_pp = scaled_sys - scaled_dia

        # Build full qt input row in exact training order
        qt_input = np.array([[
            scaled_age,   # Age     — fixed from raw dict, never mutated by DiCE
            scaled_hr,    # Heart_Rate_bpm
            scaled_sys,   # Systolic_BP
            scaled_dia,   # Diastolic_BP
            scaled_trop,  # hs_Troponin_I_ngL
            scaled_pp,    # Pulse_Pressure (approximated)
            scaled_aat    # Age_Adjusted_Troponin
        ]])

        inv = qt.inverse_transform(qt_input)[0]
        _, inv_hr, inv_sys, inv_dia, inv_trop, _, inv_aat = inv

        # Blood_Glucose & Serum_Creatinine — never quantile-transformed, pass through
        bg  = get_val(row_dict, "Blood_Glucose_mmolL",   orig_row["Blood_Glucose_mmolL"])
        scr = get_val(row_dict, "Serum_Creatinine_mgdL", orig_row["Serum_Creatinine_mgdL"])

        # Chest_Pain — LabelEncoder decode: 0/1 → No/Yes
        cp_enc = get_val(row_dict, "Chest_Pain", orig_row["Chest_Pain"])
        cp_le  = encoders.get("Chest_Pain", None)
        if cp_le:
            try:
                cp_label = cp_le.inverse_transform([int(round(cp_enc))])[0]
            except Exception:
                cp_label = "Yes" if cp_enc >= 0.5 else "No"
        else:
            cp_label = "Yes" if cp_enc >= 0.5 else "No"

        # Weighted scores — engineered integer sums, report as-is
        wrs = get_val(row_dict, "Weighted_Risk_Score",    orig_row["Weighted_Risk_Score"])
        wss = get_val(row_dict, "Weighted_Symptom_Score", orig_row["Weighted_Symptom_Score"])

        return {
            "Blood_Glucose_mmolL":    round(bg,       2),
            "Systolic_BP (mmHg)":     round(inv_sys,  1),
            "Diastolic_BP (mmHg)":    round(inv_dia,  1),
            "Heart_Rate_bpm":         round(inv_hr,   1),
            "hs_Troponin_I_ngL":      round(inv_trop, 4),
            "Serum_Creatinine_mgdL":  round(scr,      2),
            "Chest_Pain":             cp_label,
            "Weighted_Risk_Score":    round(wrs,      1),
            "Weighted_Symptom_Score": round(wss,      1),
            "Age_Adjusted_Troponin":  round(inv_aat,  3),
        }

    # ------------------------------------------------------------------
    # F. Build original decoded values from raw patient dict for comparison
    # ------------------------------------------------------------------
    sys_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[0])
    dia_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[1])

    def bin_encode(val):
        return 1 if str(val).strip().lower() in ["yes", "1", "true"] else 0

    wss_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in SYMPTOM_WEIGHTS.items())
    wrs_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in RISK_WEIGHTS.items())
    aat_orig = age_raw * np.log1p(float(raw_patient_dict["hs_Troponin_I_ngL"]))

    original_decoded = {
        "Blood_Glucose_mmolL":    raw_patient_dict["Blood_Glucose_mmolL"],
        "Systolic_BP (mmHg)":     sys_raw,
        "Diastolic_BP (mmHg)":    dia_raw,
        "Heart_Rate_bpm":         raw_patient_dict["Heart_Rate_bpm"],
        "hs_Troponin_I_ngL":      raw_patient_dict["hs_Troponin_I_ngL"],
        "Serum_Creatinine_mgdL":  raw_patient_dict["Serum_Creatinine_mgdL"],
        "Chest_Pain":             raw_patient_dict.get("Chest_Pain", "Unknown"),
        "Weighted_Risk_Score":    wrs_orig,
        "Weighted_Symptom_Score": wss_orig,
        "Age_Adjusted_Troponin":  round(aat_orig, 3),
    }

    # ------------------------------------------------------------------
    # G. Decode every CF row and build comparison records
    # ------------------------------------------------------------------
    records = []
    for i, (_, cf_row) in enumerate(cf_df.iterrows()):
        decoded = inverse_transform_row(cf_row.to_dict())

        for feature, orig_val in original_decoded.items():
            cf_val = decoded[feature]

            if feature == "Chest_Pain":
                changed     = orig_val != cf_val
                delta       = f"{orig_val} → {cf_val}" if changed else "—"
                changed_flag = "✅" if changed else "—"
            else:
                try:
                    diff = float(cf_val) - float(orig_val)
                    if abs(diff) < 1e-3:
                        delta        = "—"
                        changed_flag = "—"
                    else:
                        arrow        = "↑" if diff > 0 else "↓"
                        delta        = f"{arrow} {abs(diff):.3f}"
                        changed_flag = "✅"
                except Exception:
                    delta        = "—"
                    changed_flag = "—"

            records.append({
                "CF #":           i + 1,
                "Feature":        feature,
                "Original":       orig_val,
                "Counterfactual": cf_val,
                "Change":         delta,
                "Modified":       changed_flag,
            })

    summary_df = pd.DataFrame(records)

    # ------------------------------------------------------------------
    # H. Print clean per-CF change tables
    # ------------------------------------------------------------------
    print("\n" + "="*80)
    print(" DECODED COUNTERFACTUAL PATHWAYS — CLINICAL UNITS")
    print("="*80)

    for cf_num in summary_df["CF #"].unique():
        subset       = summary_df[summary_df["CF #"] == cf_num]
        only_changes = subset[subset["Change"] != "—"][["Feature", "Original", "Counterfactual", "Change"]]
        print(f"\n── Counterfactual Pathway #{cf_num} (changed features only) ──")
        if only_changes.empty:
            print("  No feature changes detected.")
        else:
            print(only_changes.to_string(index=False))

    return summary_df


# ==============================================================================
# 6. RUN DECODER ON GENERATED COUNTERFACTUALS
# ==============================================================================
decoded_summary = decode_counterfactuals(
    cf_object        = cf,
    raw_patient_dict = sample_raw_patient,
    artifacts_dir    = "."   # Change to "/kaggle/working" if joblibs saved there
)


 SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS...


100%|██████████| 1/1 [02:12<00:00, 132.50s/it]

Query instance (original outcome : 2)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,8.6,-0.862544,1.0,3.0,-1.199766,-1.634747,5.0,1.03,-1.629992,-1.613814,2



Diverse Counterfactual set (new outcome: 0.0)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,11.6,0.64563078,0.0,-,-0.4459191,0.3057963,-,1.1,1.484238386,0.7684021,0.0
0,13.5,0.53798038,-,4.0,0.0803789,1.82455599,-,1.09,0.393742323,0.501496375,0.0
0,17.0,-0.23679888,-,4.0,0.3361779,1.38790762,-,1.32,0.480111331,0.547256052,0.0
0,11.6,0.64563078,-,5.0,0.6059551,0.3057963,-,1.1,0.636427462,0.7684021,0.0
0,9.7,-0.63775767,-,3.1,0.2374111,-0.2,1.8,1.06,0.515058479,1.0,0.0
0,12.0,0.09801306,-,4.0,1.0618842,-0.23164105,-,1.62,0.549896002,0.534730136,0.0
0,9.2,1.48653698,-,4.0,1.1844468,0.71846199,-,0.97,1.182138773,0.83261013,0.0
0,5.7,-0.1093651,-,-,-0.4459191,0.21620388,6.0,1.46,1.484238386,1.423801184,0.0
0,9.2,1.48653698,-,4.0,1.1844468,0.71846199,-,1.25,0.922708511,0.83261013,0.0
0,11.7,0.31896937,-,4.0,-0.0564856,0.40199503,-,1.69,0.884240866,0.802254558,0.0



 DECODED COUNTERFACTUAL PATHWAYS — CLINICAL UNITS

── Counterfactual Pathway #1 (changed features only) ──
              Feature Original Counterfactual   Change
  Blood_Glucose_mmolL      8.6           11.6  ↑ 3.000
   Systolic_BP (mmHg)    124.0          148.0 ↑ 24.000
  Diastolic_BP (mmHg)     85.0           99.0 ↑ 14.000
       Heart_Rate_bpm       81          109.0 ↑ 28.000
    hs_Troponin_I_ngL   0.0063          0.825  ↑ 0.819
Serum_Creatinine_mgdL     1.03            1.1  ↑ 0.070
           Chest_Pain      Yes             No Yes → No
Age_Adjusted_Troponin    0.276          22.79 ↑ 22.514

── Counterfactual Pathway #2 (changed features only) ──
              Feature Original Counterfactual   Change
  Blood_Glucose_mmolL      8.6           13.5  ↑ 4.900
   Systolic_BP (mmHg)    124.0          156.0 ↑ 32.000
  Diastolic_BP (mmHg)     85.0           98.0 ↑ 13.000
       Heart_Rate_bpm       81          132.0 ↑ 51.000
    hs_Troponin_I_ngL   0.0063         0.2255  ↑ 0.219
Serum_Crea

# Acute to subacute

In [8]:
import dice_ml
import pandas as pd
import numpy as np
import torch
import joblib

# ==============================================================================
# 1. OPTIMIZED BATCHED PREDICTION FUNCTION (GPU PARALLELIZATION)
# ==============================================================================
def dice_transformer_predict_fn(input_df, batch_size=64):
    """
    Optimized black-box bridge function. Compiles all candidate rows, tokenizes them 
    in parallel blocks, and utilizes GPU batch processing to eliminate the 
    serial loop bottleneck.
    """
    model.eval()
    
    selected_features = [
        "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
        "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
        "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
    ]
    
    # 1. Fast serialization of all rows into target text pipe format
    texts = [
        " | ".join([f"{col}:{float(row[col]):.4f}" for col in selected_features])
        for _, row in input_df.iterrows()
    ]
    
    if not texts:
        return np.empty((0, len(CLASS_NAMES)))
        
    probabilities_list = []
    
    # 2. Process text records in parallel mini-batches rather than row-by-row
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        
        # Batched tokenization: processes multiple patients simultaneously
        inputs = tokenizer(
            batch_texts, 
            padding="max_length", 
            truncation=True,
            max_length=256, 
            return_tensors="pt"
        ).to(DEVICE)
        
        # Batched forward pass
        with torch.no_grad():
            outputs = model(**inputs)
            prob = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            probabilities_list.append(prob)
            
    # Combine the processed blocks back into a single multi-dimensional NumPy array
    return np.vstack(probabilities_list)


# ==============================================================================
# 2. SKLEARN-COMPATIBLE WRAPPER CLASS
# ==============================================================================
class BioMedBERTWrapper:
    """
    Wraps the transformer predict function to mimic sklearn's classifier interface.
    DiCE's sklearn backend requires predict_proba() and predict() methods.
    """
    def predict_proba(self, input_df):
        if not isinstance(input_df, pd.DataFrame):
            input_df = pd.DataFrame(input_df, columns=selected_features_list)
        input_df = input_df.astype(float)
        return dice_transformer_predict_fn(input_df)
    
    def predict(self, input_df):
        proba = self.predict_proba(input_df)
        return np.argmax(proba, axis=1)


# ==============================================================================
# 3. FIND AN ACUTE PATIENT FROM THE TEST SET
# ==============================================================================
selected_features_list = [
    "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
    "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
    "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
]

# ── Target classes configuration ──────────────────────────────────────────────
SOURCE_CLASS  = "Acute"
TARGET_CLASS  = "Sub-acute"

# Pull all test rows whose ground-truth label matches our source state (Acute)
acute_test_rows = test_df[test_df["Phase"] == SOURCE_CLASS].reset_index(drop=True)

if acute_test_rows.empty:
    raise ValueError(f"No '{SOURCE_CLASS}' patients found in test_df. Check CLASS_NAMES: {CLASS_NAMES}")

# Pick the first confirmed Acute patient as the query instance
raw_acute_index = 0
acute_processed_point = acute_test_rows[selected_features_list].iloc[[raw_acute_index]].astype(float)

print(f"Query patient — ground-truth label: {SOURCE_CLASS}")
print(acute_processed_point.to_string())

# Verify current base model tracking probabilities
check_proba = dice_transformer_predict_fn(acute_processed_point)
check_pred  = CLASS_NAMES[int(np.argmax(check_proba))]
print(f"\nModel prediction: {check_pred}  (confidence: {check_proba.max()*100:.1f}%)")
print(f"Class probabilities: { {CLASS_NAMES[i]: f'{p*100:.1f}%' for i, p in enumerate(check_proba[0])} }")

if check_pred != SOURCE_CLASS:
    print(f"\n[WARNING] Model predicts '{check_pred}' not '{SOURCE_CLASS}'. "
          f"Counterfactuals will still target '{TARGET_CLASS}' but the "
          f"transition narrative may differ from expected.")


# ==============================================================================
# 4. INITIALIZE DiCE DATA AND CONFIGURATION ALIGNMENTS
# ==============================================================================
# A. Background pool — all test rows, outcome encoded as integer index
test_pool_df = test_df[selected_features_list].copy().astype(float)
test_pool_df['Phase'] = test_df['Phase'].apply(lambda x: CLASS_NAMES.index(x)).astype(int)

# B. Query — the single Acute patient, no outcome column
query_instance_df = acute_processed_point.copy()

d = dice_ml.Data(
    dataframe=test_pool_df,
    continuous_features=[col for col in selected_features_list if col != "Chest_Pain"],
    outcome_name='Phase'
)

m = dice_ml.Model(
    model=BioMedBERTWrapper(),
    backend="sklearn",
    model_type="classifier"
)

exp = dice_ml.Dice(d, m, method="genetic")


# ==============================================================================
# 5. GENERATE COUNTERFACTUAL RECIPES — Acute → Sub-acute
# ==============================================================================
print("\n" + "="*80)
print(f" SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS")
print(f" TRANSITION: {SOURCE_CLASS.upper()} → {TARGET_CLASS.upper()}")
print("="*80)

target_class_index = CLASS_NAMES.index(TARGET_CLASS)

cf = exp.generate_counterfactuals(
    query_instances=query_instance_df,
    total_CFs=15,
    desired_class=target_class_index,
    proximity_weight=0.5,
    diversity_weight=1.5
)

cf.visualize_as_dataframe(show_only_changes=True)


# ==============================================================================
# 6. COUNTERFACTUAL DECODER — Reverses preprocessing back to clinical units
# ==============================================================================
def decode_counterfactuals(cf_object, raw_patient_dict, artifacts_dir="."):
    qt       = joblib.load(f"{artifacts_dir}/fitted_quantile_transformer.joblib")
    encoders = joblib.load(f"{artifacts_dir}/fitted_categorical_encoders.joblib")

    continuous_layout = [
        'Age', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP',
        'hs_Troponin_I_ngL', 'Pulse_Pressure', 'Age_Adjusted_Troponin'
    ]

    SYMPTOM_WEIGHTS = {'Chest_Pain': 4, 'Shortness_of_Breath': 1, 'Nausea': 1, 'Sweating': 1}
    RISK_WEIGHTS    = {'Prior_MI': 1, 'Hypertension': 1, 'Diabetes_Mellitus': 1,
                       'Smoking': 1, 'Hyperlipidaemia': 1, 'Family_History_CAD': 1}

    cf_df    = cf_object.cf_examples_list[0].final_cfs_df.copy()
    orig_df  = cf_object.cf_examples_list[0].test_instance_df.copy()
    orig_row = orig_df[selected_features_list].iloc[0]

    age_raw    = float(raw_patient_dict["Age"])
    scaled_age = qt.transform(
        pd.DataFrame([[age_raw, 0, 0, 0, 0, 0, 0]], columns=continuous_layout)
    )[0][0]

    def get_val(row_dict, col, fallback):
        v = row_dict.get(col, None)
        if v is None:
            return float(fallback)
        if isinstance(v, float) and np.isnan(v):
            return float(fallback)
        try:
            f = float(v)
            return float(fallback) if np.isnan(f) else f
        except (ValueError, TypeError):
            return float(fallback)

    def inverse_transform_row(row_dict):
        scaled_sys  = get_val(row_dict, "Systolic_BP",           orig_row["Systolic_BP"])
        scaled_dia  = get_val(row_dict, "Diastolic_BP",          orig_row["Diastolic_BP"])
        scaled_hr   = get_val(row_dict, "Heart_Rate_bpm",        orig_row["Heart_Rate_bpm"])
        scaled_trop = get_val(row_dict, "hs_Troponin_I_ngL",     orig_row["hs_Troponin_I_ngL"])
        scaled_aat  = get_val(row_dict, "Age_Adjusted_Troponin", orig_row["Age_Adjusted_Troponin"])
        scaled_pp   = scaled_sys - scaled_dia

        qt_input = np.array([[scaled_age, scaled_hr, scaled_sys, scaled_dia,
                               scaled_trop, scaled_pp, scaled_aat]])
        inv = qt.inverse_transform(qt_input)[0]
        _, inv_hr, inv_sys, inv_dia, inv_trop, _, inv_aat = inv

        bg  = get_val(row_dict, "Blood_Glucose_mmolL",   orig_row["Blood_Glucose_mmolL"])
        scr = get_val(row_dict, "Serum_Creatinine_mgdL", orig_row["Serum_Creatinine_mgdL"])

        cp_enc = get_val(row_dict, "Chest_Pain", orig_row["Chest_Pain"])
        cp_le  = encoders.get("Chest_Pain", None)
        if cp_le:
            try:
                cp_label = cp_le.inverse_transform([int(round(cp_enc))])[0]
            except Exception:
                cp_label = "Yes" if cp_enc >= 0.5 else "No"
        else:
            cp_label = "Yes" if cp_enc >= 0.5 else "No"

        wrs = get_val(row_dict, "Weighted_Risk_Score",    orig_row["Weighted_Risk_Score"])
        wss = get_val(row_dict, "Weighted_Symptom_Score", orig_row["Weighted_Symptom_Score"])

        return {
            "Blood_Glucose_mmolL":    round(bg,       2),
            "Systolic_BP (mmHg)":     round(inv_sys,  1),
            "Diastolic_BP (mmHg)":    round(inv_dia,  1),
            "Heart_Rate_bpm":         round(inv_hr,   1),
            "hs_Troponin_I_ngL":      round(inv_trop, 4),
            "Serum_Creatinine_mgdL":  round(scr,      2),
            "Chest_Pain":             cp_label,
            "Weighted_Risk_Score":    round(wrs,      1),
            "Weighted_Symptom_Score": round(wss,      1),
            "Age_Adjusted_Troponin":  round(inv_aat,  3),
        }

    # Original patient values decoded from raw dict
    sys_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[0])
    dia_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[1])

    def bin_encode(val):
        return 1 if str(val).strip().lower() in ["yes", "1", "true"] else 0

    wss_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in SYMPTOM_WEIGHTS.items())
    wrs_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in RISK_WEIGHTS.items())
    aat_orig = age_raw * np.log1p(float(raw_patient_dict["hs_Troponin_I_ngL"]))

    original_decoded = {
        "Blood_Glucose_mmolL":    raw_patient_dict["Blood_Glucose_mmolL"],
        "Systolic_BP (mmHg)":     sys_raw,
        "Diastolic_BP (mmHg)":    dia_raw,
        "Heart_Rate_bpm":         raw_patient_dict["Heart_Rate_bpm"],
        "hs_Troponin_I_ngL":      raw_patient_dict["hs_Troponin_I_ngL"],
        "Serum_Creatinine_mgdL":  raw_patient_dict["Serum_Creatinine_mgdL"],
        "Chest_Pain":             raw_patient_dict.get("Chest_Pain", "Unknown"),
        "Weighted_Risk_Score":    wrs_orig,
        "Weighted_Symptom_Score": wss_orig,
        "Age_Adjusted_Troponin":  round(aat_orig, 3),
    }

    records = []
    for i, (_, cf_row) in enumerate(cf_df.iterrows()):
        decoded = inverse_transform_row(cf_row.to_dict())
        for feature, orig_val in original_decoded.items():
            cf_val = decoded[feature]
            if feature == "Chest_Pain":
                changed      = orig_val != cf_val
                delta        = f"{orig_val} → {cf_val}" if changed else "—"
                changed_flag = "✅" if changed else "—"
            else:
                try:
                    diff = float(cf_val) - float(orig_val)
                    if abs(diff) < 1e-3:
                        delta        = "—"
                        changed_flag = "—"
                    else:
                        arrow        = "↑" if diff > 0 else "↓"
                        delta        = f"{arrow} {abs(diff):.3f}"
                        changed_flag = "✅"
                except Exception:
                    delta        = "—"
                    changed_flag = "—"

            records.append({
                "CF #":           i + 1,
                "Feature":         feature,
                "Original":        orig_val,
                "Counterfactual":  cf_val,
                "Change":          delta,
                "Modified":        changed_flag,
            })

    summary_df = pd.DataFrame(records)

    print("\n" + "="*80)
    print(f" DECODED COUNTERFACTUAL PATHWAYS — {SOURCE_CLASS.upper()} → {TARGET_CLASS.upper()}")
    print(" CLINICAL UNITS")
    print("="*80)

    for cf_num in summary_df["CF #"].unique():
        subset       = summary_df[summary_df["CF #"] == cf_num]
        only_changes = subset[subset["Change"] != "—"][["Feature", "Original", "Counterfactual", "Change"]]
        print(f"\n── Counterfactual Pathway #{cf_num} (changed features only) ──")
        if only_changes.empty:
            print("  No feature changes detected.")
        else:
            print(only_changes.to_string(index=False))

    return summary_df


# ==============================================================================
# 7. RUN DECODER
# ==============================================================================
# Clinical units representing a hyper-acute presentation profile baseline
sample_acute_patient = {
    "Age":                  acute_test_rows.iloc[raw_acute_index].get("Age", 72),
    "Blood_Pressure_mmHg":  "172/102",     # Extreme Hypertensive Crisis
    "Heart_Rate_bpm":        acute_test_rows.iloc[raw_acute_index].get("Heart_Rate_bpm", 112),
    "hs_Troponin_I_ngL":    acute_test_rows.iloc[raw_acute_index].get("hs_Troponin_I_ngL", 0.2918), # Elevated acute leak
    "Blood_Glucose_mmolL":  acute_test_rows.iloc[raw_acute_index].get("Blood_Glucose_mmolL", 4.6),
    "Serum_Creatinine_mgdL":acute_test_rows.iloc[raw_acute_index].get("Serum_Creatinine_mgdL", 2.46),
    "Chest_Pain":           "Yes",
    "Shortness_of_Breath":  "Yes",          # Severe emergency presentation
    "Nausea":               "Yes",
    "Sweating":             "No",
    "Hypertension":         "Yes",
    "Diabetes_Mellitus":    "Yes",
    "Smoking":              "Yes",
    "Hyperlipidaemia":      "Yes",
    "Prior_MI":             "No",         
    "Family_History_CAD":   "Yes",
}

decoded_summary = decode_counterfactuals(
    cf_object        = cf,
    raw_patient_dict = sample_acute_patient,
    artifacts_dir    = "."
)

Query patient — ground-truth label: Acute
   Blood_Glucose_mmolL  Diastolic_BP  Chest_Pain  Weighted_Risk_Score  Systolic_BP  Heart_Rate_bpm  Weighted_Symptom_Score  Serum_Creatinine_mgdL  hs_Troponin_I_ngL  Age_Adjusted_Troponin
0                 18.0     -0.376283         1.0                  4.0     0.855287        -0.06277                     7.0                   2.17           0.843771               0.626808

Model prediction: Acute  (confidence: 99.3%)
Class probabilities: {'Acute': '99.3%', 'Chronic': '0.1%', 'Non-MI': '0.1%', 'Sub-acute': '0.5%'}

 SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS
 TRANSITION: ACUTE → SUB-ACUTE


100%|██████████| 1/1 [29:39<00:00, 1779.03s/it]

Query instance (original outcome : 0)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,18.0,-0.376283,1.0,4.0,0.855287,-0.06277,7.0,2.17,0.843771,0.626808,0



Diverse Counterfactual set (new outcome: 3.0)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,17.8,0.64563078,-,-,0.9574536,-0.39249203,-,1.92,0.469663054,0.355822623,3.0
0,14.1,0.09801306,-,-,0.1422445,0.47661874,-,1.96,0.225292385,0.245811716,3.0
0,5.3,-0.23679888,-,-,-,0.40199503,6.0,1.75,0.132113025,0.152776539,3.0
0,11.4,0.31896937,-,-,1.1844468,0.21620388,-,1.56,0.415808856,0.344447702,3.0
0,17.8,0.09801306,-,-,-0.4459191,0.21620388,5.0,1.91,0.208353221,0.188849837,3.0
0,13.7,-0.23679888,-,5.0,-0.241963,0.81090987,6.0,2.09,0.242622882,0.271948457,3.0
0,6.1,-0.51922536,-,-,1.5135889,-0.31633037,6.0,1.45,0.332458287,0.2759642,3.0
0,12.2,0.20978712,-,5.0,-0.3695582,0.81090987,6.0,2.19,0.168825939,0.178752512,3.0
0,6.1,-0.51922536,-,2.2,0.6919648,-0.31633037,5.9,1.93,0.332458287,0.2759642,3.0
0,14.5,-0.31436273,-,0.0,0.4049968,0.5,5.4,2.06,1.009891648,0.0,3.0



 DECODED COUNTERFACTUAL PATHWAYS — ACUTE → SUB-ACUTE
 CLINICAL UNITS

── Counterfactual Pathway #1 (changed features only) ──
               Feature  Original Counterfactual    Change
   Blood_Glucose_mmolL      18.0           17.8   ↓ 0.200
    Systolic_BP (mmHg)     172.0          168.0   ↓ 4.000
   Diastolic_BP (mmHg)     102.0           99.0   ↓ 3.000
        Heart_Rate_bpm  -0.06277          101.0 ↑ 101.063
     hs_Troponin_I_ngL  0.843771         0.2601   ↓ 0.584
 Serum_Creatinine_mgdL      2.17           1.92   ↓ 0.250
   Weighted_Risk_Score         5            4.0   ↓ 1.000
Weighted_Symptom_Score         6            7.0   ↑ 1.000
 Age_Adjusted_Troponin    -0.614         12.253  ↑ 12.867

── Counterfactual Pathway #2 (changed features only) ──
               Feature  Original Counterfactual    Change
   Blood_Glucose_mmolL      18.0           14.1   ↓ 3.900
    Systolic_BP (mmHg)     172.0          157.0  ↓ 15.000
   Diastolic_BP (mmHg)     102.0           94.0   ↓ 8.000
    

# sub acute to chronic

In [9]:
import dice_ml
import pandas as pd
import numpy as np
import torch
import joblib

# ==============================================================================
# 1. PREDICTION FUNCTION WITH ROBUST TYPE CASTING
# ==============================================================================
def dice_transformer_predict_fn(input_df):
    """
    Acts as a black-box bridge. DiCE inputs a tabular DataFrame,
    isolates the 10 features, serializes them to text, and runs BioMedBERT.
    """
    model.eval()
    
    selected_features = [
        "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
        "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
        "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
    ]
    
    probabilities_list = []
    for _, row in input_df.iterrows():
        serialized_text = " | ".join([f"{col}:{float(row[col]):.4f}" for col in selected_features])
        inputs = tokenizer(
            serialized_text, padding="max_length", truncation=True,
            max_length=256, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
            prob = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
            probabilities_list.append(prob)
    return np.array(probabilities_list)


# ==============================================================================
# 2. SKLEARN-COMPATIBLE WRAPPER CLASS
# ==============================================================================
class BioMedBERTWrapper:
    def predict_proba(self, input_df):
        if not isinstance(input_df, pd.DataFrame):
            input_df = pd.DataFrame(input_df, columns=selected_features_list)
        input_df = input_df.astype(float)
        return dice_transformer_predict_fn(input_df)
    
    def predict(self, input_df):
        proba = self.predict_proba(input_df)
        return np.argmax(proba, axis=1)


# ==============================================================================
# 3. FIND A SUB-ACUTE PATIENT FROM THE TEST SET
# ==============================================================================
selected_features_list = [
    "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
    "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score", 
    "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin"
]

# ── Source transition: Sub-acute → Chronic ────────────────────────────────────
SOURCE_CLASS  = "Sub-acute"
TARGET_CLASS  = "Chronic"

# Pull all test rows whose ground-truth label is Sub-acute
subacute_test_rows = test_df[test_df["Phase"] == SOURCE_CLASS].reset_index(drop=True)

if subacute_test_rows.empty:
    raise ValueError(f"No '{SOURCE_CLASS}' patients found in test_df. Check CLASS_NAMES: {CLASS_NAMES}")

# Pick the first confirmed Sub-acute patient as the query instance
raw_subacute_index = 0
subacute_processed_point = subacute_test_rows[selected_features_list].iloc[[raw_subacute_index]].astype(float)

print(f"Query patient — ground-truth label: {SOURCE_CLASS}")
print(subacute_processed_point.to_string())

# Verify the model also predicts this patient as Sub-acute
check_proba = dice_transformer_predict_fn(subacute_processed_point)
check_pred  = CLASS_NAMES[int(np.argmax(check_proba))]
print(f"\nModel prediction: {check_pred}  (confidence: {check_proba.max()*100:.1f}%)")
print(f"Class probabilities: { {CLASS_NAMES[i]: f'{p*100:.1f}%' for i, p in enumerate(check_proba[0])} }")

if check_pred != SOURCE_CLASS:
    print(f"\n[WARNING] Model predicts '{check_pred}' not '{SOURCE_CLASS}'. "
          f"Counterfactuals will still target '{TARGET_CLASS}' but the "
          f"transition narrative may differ from expected.")


# ==============================================================================
# 4. INITIALIZE DATA AND CONFIGURATION ALIGNMENTS
# ==============================================================================
# A. Background pool — all test rows, outcome encoded as integer index
test_pool_df = test_df[selected_features_list].copy().astype(float)
test_pool_df['Phase'] = test_df['Phase'].apply(lambda x: CLASS_NAMES.index(x)).astype(int)

# B. Query — the single Sub-acute patient, no outcome column
query_instance_df = subacute_processed_point.copy()

d = dice_ml.Data(
    dataframe=test_pool_df,
    continuous_features=[col for col in selected_features_list if col != "Chest_Pain"],
    outcome_name='Phase'
)

m = dice_ml.Model(
    model=BioMedBERTWrapper(),
    backend="sklearn",
    model_type="classifier"
)

exp = dice_ml.Dice(d, m, method="genetic")


# ==============================================================================
# 5. GENERATE COUNTERFACTUAL RECIPES — Sub-acute → Chronic
# ==============================================================================
print("\n" + "="*80)
print(f" SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS")
print(f" TRANSITION: {SOURCE_CLASS.upper()} → {TARGET_CLASS.upper()}")
print("="*80)

target_class_index = CLASS_NAMES.index(TARGET_CLASS)

cf = exp.generate_counterfactuals(
    query_instances=query_instance_df,
    total_CFs=15,
    desired_class=target_class_index,
    proximity_weight=0.5,
    diversity_weight=1.5
)

cf.visualize_as_dataframe(show_only_changes=True)


# ==============================================================================
# 6. COUNTERFACTUAL DECODER — Reverses preprocessing back to clinical units
# ==============================================================================
def decode_counterfactuals(cf_object, raw_patient_dict, artifacts_dir="."):
    qt       = joblib.load(f"{artifacts_dir}/fitted_quantile_transformer.joblib")
    encoders = joblib.load(f"{artifacts_dir}/fitted_categorical_encoders.joblib")

    continuous_layout = [
        'Age', 'Heart_Rate_bpm', 'Systolic_BP', 'Diastolic_BP',
        'hs_Troponin_I_ngL', 'Pulse_Pressure', 'Age_Adjusted_Troponin'
    ]

    SYMPTOM_WEIGHTS = {'Chest_Pain': 4, 'Shortness_of_Breath': 1, 'Nausea': 1, 'Sweating': 1}
    RISK_WEIGHTS    = {'Prior_MI': 1, 'Hypertension': 1, 'Diabetes_Mellitus': 1,
                       'Smoking': 1, 'Hyperlipidaemia': 1, 'Family_History_CAD': 1}

    cf_df    = cf_object.cf_examples_list[0].final_cfs_df.copy()
    orig_df  = cf_object.cf_examples_list[0].test_instance_df.copy()
    orig_row = orig_df[selected_features_list].iloc[0]

    age_raw    = float(raw_patient_dict["Age"])
    scaled_age = qt.transform(
        pd.DataFrame([[age_raw, 0, 0, 0, 0, 0, 0]], columns=continuous_layout)
    )[0][0]

    def get_val(row_dict, col, fallback):
        v = row_dict.get(col, None)
        if v is None:
            return float(fallback)
        if isinstance(v, float) and np.isnan(v):
            return float(fallback)
        try:
            f = float(v)
            return float(fallback) if np.isnan(f) else f
        except (ValueError, TypeError):
            return float(fallback)

    def inverse_transform_row(row_dict):
        scaled_sys  = get_val(row_dict, "Systolic_BP",           orig_row["Systolic_BP"])
        scaled_dia  = get_val(row_dict, "Diastolic_BP",          orig_row["Diastolic_BP"])
        scaled_hr   = get_val(row_dict, "Heart_Rate_bpm",        orig_row["Heart_Rate_bpm"])
        scaled_trop = get_val(row_dict, "hs_Troponin_I_ngL",     orig_row["hs_Troponin_I_ngL"])
        scaled_aat  = get_val(row_dict, "Age_Adjusted_Troponin", orig_row["Age_Adjusted_Troponin"])
        scaled_pp   = scaled_sys - scaled_dia

        qt_input = np.array([[scaled_age, scaled_hr, scaled_sys, scaled_dia,
                               scaled_trop, scaled_pp, scaled_aat]])
        inv = qt.inverse_transform(qt_input)[0]
        _, inv_hr, inv_sys, inv_dia, inv_trop, _, inv_aat = inv

        bg  = get_val(row_dict, "Blood_Glucose_mmolL",   orig_row["Blood_Glucose_mmolL"])
        scr = get_val(row_dict, "Serum_Creatinine_mgdL", orig_row["Serum_Creatinine_mgdL"])

        cp_enc = get_val(row_dict, "Chest_Pain", orig_row["Chest_Pain"])
        cp_le  = encoders.get("Chest_Pain", None)
        if cp_le:
            try:
                cp_label = cp_le.inverse_transform([int(round(cp_enc))])[0]
            except Exception:
                cp_label = "Yes" if cp_enc >= 0.5 else "No"
        else:
            cp_label = "Yes" if cp_enc >= 0.5 else "No"

        wrs = get_val(row_dict, "Weighted_Risk_Score",    orig_row["Weighted_Risk_Score"])
        wss = get_val(row_dict, "Weighted_Symptom_Score", orig_row["Weighted_Symptom_Score"])

        return {
            "Blood_Glucose_mmolL":    round(bg,       2),
            "Systolic_BP (mmHg)":     round(inv_sys,  1),
            "Diastolic_BP (mmHg)":    round(inv_dia,  1),
            "Heart_Rate_bpm":         round(inv_hr,   1),
            "hs_Troponin_I_ngL":      round(inv_trop, 4),
            "Serum_Creatinine_mgdL":  round(scr,      2),
            "Chest_Pain":             cp_label,
            "Weighted_Risk_Score":    round(wrs,      1),
            "Weighted_Symptom_Score": round(wss,      1),
            "Age_Adjusted_Troponin":  round(inv_aat,  3),
        }

    # Original patient values decoded from raw dict
    sys_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[0])
    dia_raw = float(str(raw_patient_dict["Blood_Pressure_mmHg"]).split('/')[1])

    def bin_encode(val):
        return 1 if str(val).strip().lower() in ["yes", "1", "true"] else 0

    wss_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in SYMPTOM_WEIGHTS.items())
    wrs_orig = sum(bin_encode(raw_patient_dict.get(c, 0)) * w for c, w in RISK_WEIGHTS.items())
    aat_orig = age_raw * np.log1p(float(raw_patient_dict["hs_Troponin_I_ngL"]))

    original_decoded = {
        "Blood_Glucose_mmolL":    raw_patient_dict["Blood_Glucose_mmolL"],
        "Systolic_BP (mmHg)":     sys_raw,
        "Diastolic_BP (mmHg)":    dia_raw,
        "Heart_Rate_bpm":         raw_patient_dict["Heart_Rate_bpm"],
        "hs_Troponin_I_ngL":      raw_patient_dict["hs_Troponin_I_ngL"],
        "Serum_Creatinine_mgdL":  raw_patient_dict["Serum_Creatinine_mgdL"],
        "Chest_Pain":             raw_patient_dict.get("Chest_Pain", "Unknown"),
        "Weighted_Risk_Score":    wrs_orig,
        "Weighted_Symptom_Score": wss_orig,
        "Age_Adjusted_Troponin":  round(aat_orig, 3),
    }

    records = []
    for i, (_, cf_row) in enumerate(cf_df.iterrows()):
        decoded = inverse_transform_row(cf_row.to_dict())
        for feature, orig_val in original_decoded.items():
            cf_val = decoded[feature]
            if feature == "Chest_Pain":
                changed      = orig_val != cf_val
                delta        = f"{orig_val} → {cf_val}" if changed else "—"
                changed_flag = "✅" if changed else "—"
            else:
                try:
                    diff = float(cf_val) - float(orig_val)
                    if abs(diff) < 1e-3:
                        delta        = "—"
                        changed_flag = "—"
                    else:
                        arrow        = "↑" if diff > 0 else "↓"
                        delta        = f"{arrow} {abs(diff):.3f}"
                        changed_flag = "✅"
                except Exception:
                    delta        = "—"
                    changed_flag = "—"

            records.append({
                "CF #":           i + 1,
                "Feature":         feature,
                "Original":        orig_val,
                "Counterfactual":  cf_val,
                "Change":          delta,
                "Modified":        changed_flag,
            })

    summary_df = pd.DataFrame(records)

    print("\n" + "="*80)
    print(f" DECODED COUNTERFACTUAL PATHWAYS — {SOURCE_CLASS.upper()} → {TARGET_CLASS.upper()}")
    print(" CLINICAL UNITS")
    print("="*80)

    for cf_num in summary_df["CF #"].unique():
        subset       = summary_df[summary_df["CF #"] == cf_num]
        only_changes = subset[subset["Change"] != "—"][["Feature", "Original", "Counterfactual", "Change"]]
        print(f"\n── Counterfactual Pathway #{cf_num} (changed features only) ──")
        if only_changes.empty:
            print("  No feature changes detected.")
        else:
            print(only_changes.to_string(index=False))

    return summary_df


# ==============================================================================
# 7. RUN DECODER
# ==============================================================================
# Clinical units representing a patient in the intermediate Sub-acute recovery state
sample_subacute_patient = {
    "Age":                  subacute_test_rows.iloc[raw_subacute_index].get("Age", 64),
    "Blood_Pressure_mmHg":  "149/91",      # High-normal/Mild hypertension, typical post-crisis stabilization
    "Heart_Rate_bpm":        subacute_test_rows.iloc[raw_subacute_index].get("Heart_Rate_bpm", 116),
    "hs_Troponin_I_ngL":    subacute_test_rows.iloc[raw_subacute_index].get("hs_Troponin_I_ngL", 0.0296), # Plateaued/slowly clearing enzyme leak
    "Blood_Glucose_mmolL":  subacute_test_rows.iloc[raw_subacute_index].get("Blood_Glucose_mmolL", 12.7),
    "Serum_Creatinine_mgdL":subacute_test_rows.iloc[raw_subacute_index].get("Serum_Creatinine_mgdL", 1.71),
    "Chest_Pain":           "Yes",
    "Shortness_of_Breath":  "No",          # Resolving immediate emergency symptoms
    "Nausea":               "No",
    "Sweating":             "No",
    "Hypertension":         "Yes",
    "Diabetes_Mellitus":    "Yes",
    "Smoking":              "Yes",
    "Hyperlipidaemia":      "Yes",
    "Prior_MI":             "Yes",         # Marked True to reflect the recent acute event history
    "Family_History_CAD":   "Yes",
}

decoded_summary = decode_counterfactuals(
    cf_object        = cf,
    raw_patient_dict = sample_subacute_patient,
    artifacts_dir    = "."
)

Query patient — ground-truth label: Sub-acute
   Blood_Glucose_mmolL  Diastolic_BP  Chest_Pain  Weighted_Risk_Score  Systolic_BP  Heart_Rate_bpm  Weighted_Symptom_Score  Serum_Creatinine_mgdL  hs_Troponin_I_ngL  Age_Adjusted_Troponin
0                 12.2      0.209787         1.0                  5.0    -0.369558         0.81091                     6.0                   2.19           0.168826               0.178753

Model prediction: Sub-acute  (confidence: 84.0%)
Class probabilities: {'Acute': '1.5%', 'Chronic': '6.3%', 'Non-MI': '8.2%', 'Sub-acute': '84.0%'}

 SEARCHING FOR 15 DIVERSE CLINICAL COUNTERFACTUAL PATHWAYS
 TRANSITION: SUB-ACUTE → CHRONIC


100%|██████████| 1/1 [01:05<00:00, 65.19s/it]

Query instance (original outcome : 3)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,12.2,0.209787,1.0,5.0,-0.369558,0.81091,6.0,2.19,0.168826,0.178753,3



Diverse Counterfactual set (new outcome: 1.0)


,Blood_Glucose_mmolL,Diastolic_BP,Chest_Pain,Weighted_Risk_Score,Systolic_BP,Heart_Rate_bpm,Weighted_Symptom_Score,Serum_Creatinine_mgdL,hs_Troponin_I_ngL,Age_Adjusted_Troponin,Phase
0,5.6,0.77652776,-,-,0.0213293,-0.62414473,-,1.86,0.062769629,0.064028502,1.0
0,14.9,-,-,-,-0.0564856,-0.31633037,7.0,1.86,0.691464484,0.790055156,1.0
0,5.1,-0.23679888,-,4.0,0.2162039,-0.14731565,-,2.25,0.072829314,0.01355186,1.0
0,5.7,0.42797589,-,-,0.279606,0.3057963,5.0,2.02,-0.278301686,-0.117925294,1.0
0,11.6,0.953493,-,4.0,-0.5939395,-0.31633037,-,2.06,-0.380325645,-0.382032394,1.0
0,13.8,0.42797589,-,-,-0.5939395,-0.31633037,5.0,2.33,0.985546768,0.926907897,1.0
0,11.6,0.23948135,-,5.5,-0.5939395,-0.31633037,-,2.06,-0.380325645,-0.382032394,1.0
0,10.7,0.42797589,-,4.0,-0.1880329,0.12958232,-,1.75,-0.469603062,-0.483059406,1.0
0,16.9,-0.51922536,-,3.0,-0.4459191,-0.39249203,-,1.99,-0.590949059,-0.302320182,1.0
0,13.6,0.31896937,-,3.0,-,-0.51635629,-,1.42,-0.312376052,-0.440683573,1.0



 DECODED COUNTERFACTUAL PATHWAYS — SUB-ACUTE → CHRONIC
 CLINICAL UNITS

── Counterfactual Pathway #1 (changed features only) ──
               Feature  Original Counterfactual   Change
   Blood_Glucose_mmolL      12.2            5.6  ↓ 6.600
    Systolic_BP (mmHg)     149.0          155.0  ↑ 6.000
   Diastolic_BP (mmHg)      91.0          100.0  ↑ 9.000
        Heart_Rate_bpm   0.81091           97.0 ↑ 96.189
     hs_Troponin_I_ngL  0.168826         0.0331  ↓ 0.136
 Serum_Creatinine_mgdL      2.19           1.86  ↓ 0.330
   Weighted_Risk_Score         6            5.0  ↓ 1.000
Weighted_Symptom_Score         4            6.0  ↑ 2.000
 Age_Adjusted_Troponin      0.07          2.214  ↑ 2.144

── Counterfactual Pathway #2 (changed features only) ──
               Feature  Original Counterfactual    Change
   Blood_Glucose_mmolL      12.2           14.9   ↑ 2.700
    Systolic_BP (mmHg)     149.0          154.0   ↑ 5.000
   Diastolic_BP (mmHg)      91.0           95.0   ↑ 4.000
        Hear